# Versi dengan Bootstrap 95% CI (revisi Discover Informatics)

Notebook ini **sama persis** dengan notebook di repo GitHub
(`Annisa165/Extractive-Summarization-of-Indonesian-Friday-Sermons`), ditambah **Section 4C**:

1. Skor ROUGE-1/2/L dan BERTScore **per dokumen** untuk 21 dokumen test
2. **Bootstrap 95% CI** (1.000 resample, seed 42) untuk setiap metode × metrik → Tabel 3
3. **Paired bootstrap** selisih IndoBERT − baseline (untuk klaim "lebih unggul")
4. **Grouped bar chart** dengan error bar CI (pengganti radar chart, poin editor 2.1)
5. **Uji Kruskal–Wallis** ROUGE-1 per topik (poin editor 2.5)

**Cara pakai:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**.
Semua sel harus dijalankan dalam **satu sesi** (model hasil fine-tuning disimpan di `/content`).
Di akhir, file `table3_with_ci.csv`, `per_document_scores.csv`, `paired_differences.csv`,
dan `fig11_grouped_bar_ci.png` otomatis ter-download. Kirim keempat file itu.

⚠️ **Penting:** fine-tuning di GPU tidak 100% deterministik, jadi angka IndoBERT bisa sedikit
berbeda dari Tabel 3 lama. Kalau berbeda, **seluruh Tabel 3 diganti dengan angka dari run ini**
(rata-rata + CI dari run yang sama) — jangan mencampur rata-rata lama dengan CI baru.

# Extractive Summarization of Indonesian Friday Sermons Using Fine-Tuned IndoBERT and Pseudo-Labeled Data
### CORRECTED notebook -- fixes 11 issues found by actually executing the original code

**Paper**: *JISEBI (Journal of Information Systems Engineering and Business Intelligence)*
**Authors**: Supriyono, Aji Prasetya Wibawa*, Suyono, Fachrul Kurniawan

---
## What changed vs. the original notebook

| # | Issue found | Fix in this notebook |
|---|---|---|
| 1 | Sermon **topics** assigned via `np.random.choice` (commented "mock") | Real keyword-rule classifier over sermon content (Section 1B) |
| 2 | **Duration** column inflated 60x (MM:SS entered, stored/read as HH:MM) | Corrected conversion; range is now 2.5-54.5 min (was up to 3,270 min / 54.5 **hours**) |
| 3 | Pseudo-label **composite score**: TF-IDF component ~1000x smaller scale than position/keyword, so `alpha` had almost no real effect on ranking | Min-max normalize each component before combining |
| 4 | **Weight sensitivity** analysis judged configs by the average of their own score (circular, always favors the same config) | Judged by real ROUGE-1 against an external reference on a held-out 20-doc validation set |
| 5 | Pseudo-**reference** for evaluation used the *exact same function* as the TextRank baseline -> TextRank scored a trivial ~1.0 | Reference now built with a genuinely different method (LexRank via eigenvector centrality), excluded from the results table |
| 6 | Main results table (Table 3) was a **hand-typed dictionary**, not the output of the `avg_rouge()` function already defined | Table 3 now calls `avg_rouge()` on real generated summaries |
| 7 | "**BERTSum (generic)**" baseline was TF-IDF with a different ratio -- no BERT model involved | Real BERT-encoder-based extractive baseline (Section 4B, GPU) |
| 8 | "**IndoBERT**" summaries used the pseudo-label heuristic score as a *proxy*, never the fine-tuned model's own predictions | Real inference from the trained model checkpoint (Section 4B, GPU) |
| 9 | **Confusion matrix**, **topic-wise ROUGE**, **ablation study**, **compression-ratio curve** were hardcoded target numbers with cosmetic random noise added | All recomputed from real predictions/evaluation (Section 4B, GPU) |
| 10 | **BERTScore** never actually computed despite being installed and reported | Real `bert_score` call added (Section 4B, GPU) |
| 11 | All 15 auto-generated figure titles were numbered one less than their position in the manuscript (`Fig. N-1` baked into the image) | All titles corrected to match manuscript numbering (Fig. 2-16; Fig. 1 is the architecture diagram, not part of this notebook) |

**Sections 0-2 and Figures 2-6 run entirely on CPU and were executed to verify correctness** (see printed output in each cell). **Section 3 (fine-tuning) and Section 4B (real inference / real BERTSum / real BERTScore / Figures 8-16) require a GPU runtime and internet access to download `indobenchmark/indobert-base-p1` and BERTScore's underlying model** -- run those in Google Colab (Runtime -> Change runtime type -> T4 GPU).


## Section 0: Installation

In [ ]:
%%capture
!pip install transformers datasets torch scikit-learn rouge-score bert-score openpyxl networkx
!pip install matplotlib seaborn numpy pandas tqdm
print('✅ All packages installed')

In [ ]:
# Ambil dataset langsung dari repo GitHub (tidak perlu upload manual)
!git clone -q https://github.com/Annisa165/Extractive-Summarization-of-Indonesian-Friday-Sermons.git /content/repo 2>/dev/null || echo "repo sudah ada"
!cp "/content/repo/Dataset Ceramah Khutbah Jumat.xlsx" /content/Dataset_Ceramah_Khutbah_Jumat.xlsx
!ls -la /content/Dataset_Ceramah_Khutbah_Jumat.xlsx

## Section 1: Data Loading & Preprocessing
Duration conversion is corrected here (see fix #2 in the table above).

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA', 'axes.grid': True, 'grid.alpha': 0.4,
})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B',
           '#44BBA4', '#E94F37', '#393E41', '#F5A623', '#7B2D8B']

try:
    df = pd.read_excel('/content/Dataset_Ceramah_Khutbah_Jumat.xlsx', sheet_name='Youtube')
    print('✅ Dataset loaded from /content/')
except FileNotFoundError:
    from google.colab import files
    print('Please upload Dataset_Ceramah_Khutbah_Jumat.xlsx')
    uploaded = files.upload()
    df = pd.read_excel(list(uploaded.keys())[0], sheet_name='Youtube')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
def clean_transcript(text):
    """Remove YouTube caption artifacts and normalize text."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\d{2}:\d{2}:\d{2}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

def parse_viewers(v):
    if isinstance(v, (int, float)):
        return float(v)
    v = str(v).replace(',', '.').strip()
    if 'jt' in v:
        return float(v.replace('jt','')) * 1_000_000
    if 'k' in v.lower():
        return float(re.sub('[kK]', '', v)) * 1_000
    try:
        return float(v)
    except:
        return np.nan

def duration_to_minutes(d):
    """
    FIX (#2): source column has Excel format [h]:mm. Cross-checking raw
    values against video titles/content confirms durations were entered
    as MM:SS but stored/read as HH:MM -- inflating every value by 60x
    (e.g. '4:54' meant 4 min 54 sec, but is stored as 4 HOURS 54 min).
    Dividing by the extra factor of 60 (i.e. /3600 instead of /60) brings
    all 210 durations into a 2.5-54.5 minute range (mean ~22.0 min),
    matching normal Friday-sermon length. Without this fix, durations
    ranged up to 3,270 minutes (54.5 HOURS) -- root cause of Reviewer 3's
    question 'Are you sure a sermon can reach 300 minutes?': yes, and it
    was actually far worse than 300 minutes before this fix.
    """
    try:
        return d.total_seconds() / 3600
    except Exception:
        return np.nan

df['transcript_clean'] = df['Transkrip'].apply(clean_transcript)
df['word_count'] = df['transcript_clean'].apply(lambda x: len(x.split()))
df['char_count'] = df['transcript_clean'].apply(len)
df['viewers_num'] = df['Jumlah Viewers'].apply(parse_viewers)
df['duration_min'] = df['Durasi'].apply(duration_to_minutes)

print('✅ Preprocessing done')
print(f"Duration range AFTER fix: {df['duration_min'].min():.1f}-{df['duration_min'].max():.1f} min "
      f"(mean {df['duration_min'].mean():.1f} min)")
df[['word_count','char_count','viewers_num','duration_min']].describe().round(2)

## Section 1B: Topic Classification (real, content-based)
Replaces `np.random.choice` (fix #1) with a documented keyword-rule classifier.
This is still a heuristic, not human expert annotation -- **recommend a human
spot-check of a sample before final publication**, consistent with the
human-validation work already planned for Reviewer 2's evaluation-circularity
comment.

In [ ]:
TOPIC_LEXICON = {
    "Aqidah": [
        "tauhid", "aqidah", "akidah", "syirik", "musyrik", "kafir kepada",
        "rukun iman", "sifat allah", "asmaul husna", "malaikat jibril",
        "kitab-kitab allah", "hari kiamat", "qada dan qadar", "keesaan allah",
        "ilahiyah", "uluhiyah", "menyekutukan allah",
    ],
    "Fiqh": [
        "zakat", "shalat", "salat", "puasa ramadan", "puasa ramadhan", "ibadah haji",
        "berwudu", "berwudhu", "hukum wajib", "hukum sunnah", "hukum haram",
        "hukum halal", "rukun islam", "fikih", "fiqih", "hukum syariat",
        "sah atau tidak", "syarat sah", "membatalkan", "qadha shalat", "sujud",
        "rakaat", "tayamum", "nisab zakat", "zakat fitrah",
    ],
    "Akhlak": [
        "akhlak", "budi pekerti", "sopan santun", "rendah hati", "tawadhu",
        "pemaaf", "santun", "berbakti kepada orang tua", "menghormati orang tua",
        "durhaka kepada orang tua", "silaturahim", "silaturahmi", "sombong",
        "takabbur", "iri hati", "dengki",
    ],
    "Muamalah": [
        "jual beli", "riba", "berhutang", "berutang", "sedekah", "infak",
        "infaq", "harta benda", "roda ekonomi", "berbisnis", "berdagang",
        "bertransaksi", "muamalah", "warisan", "sewa menyewa", "upah kerja",
        "gaji", "pekerjaan", "pemimpin yang", "hak rakyat", "bermasyarakat",
        "gotong royong",
    ],
    "Tazkiyah": [
        "tazkiyah", "bertaubat", "taubat nasuha", "beristighfar", "muhasabah diri",
        "berhijrah menuju", "mujahadah", "hawa nafsu", "penyucian jiwa",
        "husnul khatimah", "su\'ul khatimah", "mengingat kematian",
        "menangis karena takut kepada allah", "qalbu",
    ],
}
TIE_BREAK_ORDER = ["Aqidah", "Muamalah", "Tazkiyah", "Akhlak", "Fiqh"]

def classify_topic(text):
    scores = {}
    n_tokens = max(1, len(text.split()))
    for topic, keywords in TOPIC_LEXICON.items():
        hits = sum(text.count(kw) for kw in keywords)
        scores[topic] = hits / n_tokens
    max_score = max(scores.values())
    if max_score == 0:
        return "Akhlak"
    winners = [t for t, s in scores.items() if s == max_score]
    if len(winners) == 1:
        return winners[0]
    for t in TIE_BREAK_ORDER:
        if t in winners:
            return t
    return winners[0]

df["topic"] = df["transcript_clean"].apply(classify_topic)
print("Topic distribution (keyword-rule classifier):")
print(df["topic"].value_counts())
print((df["topic"].value_counts(normalize=True) * 100).round(1))

## Figure 2: Transcript Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
n, bins, patches = ax.hist(df['word_count'], bins=25, color=PALETTE[0],
                           edgecolor='white', linewidth=0.8, alpha=0.85)
median_wc = df['word_count'].median()
ax.axvline(df['word_count'].mean(), color='red', ls='--', lw=1.8, label=f'Mean = {df["word_count"].mean():.0f}')
ax.axvline(median_wc, color='green', ls=':', lw=1.8, label=f'Median = {median_wc:.0f}')
ax.set_xlabel('Word Count per Transcript')
ax.set_ylabel('Number of Documents')
ax.set_title(f'Fig. 2 – Distribution of Transcript Word Counts (N={len(df)} Khutbah Jumat)')
ax.legend()
stats = f'μ={df["word_count"].mean():.0f}  σ={df["word_count"].std():.0f}\nMin={df["word_count"].min()}  Max={df["word_count"].max()}'
ax.text(0.97, 0.95, stats, transform=ax.transAxes, ha='right', va='top',
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig('fig02_word_count_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved')

## Figure 3: Viewer Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
v = df['viewers_num'].dropna()
axes[0].hist(v, bins=20, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Viewer Count'); axes[0].set_ylabel('Number of Videos'); axes[0].set_title('(a) Linear Scale')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
axes[1].hist(np.log10(v+1), bins=20, color=PALETTE[3], edgecolor='white', alpha=0.85)
axes[1].set_xlabel('log10(Viewer Count)'); axes[1].set_ylabel('Number of Videos'); axes[1].set_title('(b) Log Scale')
axes[1].axvline(np.log10(v.median()+1), color='navy', ls='--', lw=1.5, label=f'Median = {v.median()/1e3:.1f}K')
axes[1].legend()
fig.suptitle('Fig. 3 – Distribution of YouTube Viewer Counts for Khutbah Jumat Videos', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig03_viewer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved')

## Figure 4: Duration vs. Word Count Scatter\nNow uses the corrected duration values (fix #2) and real topic labels (fix #1).

In [ ]:
from scipy import stats as scipy_stats

mask = df['duration_min'].notna() & df['word_count'].notna()
x = df.loc[mask, 'duration_min']
y = df.loc[mask, 'word_count']
r, p = scipy_stats.pearsonr(x, y)

fig, ax = plt.subplots(figsize=(9, 6))
topic_codes = df.loc[mask, 'topic'].astype('category')
scatter = ax.scatter(x, y, c=topic_codes.cat.codes, cmap='tab10', alpha=0.7, s=60, edgecolors='white', linewidth=0.5)
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, m*x_line+b, 'r--', lw=2, label=f'Linear fit (r={r:.2f}, p<0.001)')
for i, t in enumerate(topic_codes.cat.categories):
    ax.scatter([], [], c=[plt.cm.tab10(i/10)], label=t, s=50)
ax.set_xlabel('Sermon Duration (minutes)')
ax.set_ylabel('Transcript Word Count')
ax.set_title('Fig. 4 – Duration vs. Word Count by Sermon Topic')
ax.legend(loc='upper left', fontsize=9)
ax.text(0.97, 0.05, f'r = {r:.2f}, p < 0.001', transform=ax.transAxes, ha='right',
        fontsize=11, color='red', fontweight='bold')
plt.tight_layout()
plt.savefig('fig04_duration_wordcount_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Duration range: {x.min():.1f}-{x.max():.1f} min (was up to 3,270 min before fix #2)")
print('✅ Figure 4 saved')

## Figure 5: Thematic Distribution

In [ ]:
topic_counts = df['topic'].value_counts()
colors_pie = [PALETTE[i] for i in range(len(topic_counts))]
fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    topic_counts.values, labels=topic_counts.index, autopct='%1.1f%%',
    colors=colors_pie, explode=[0.03]*len(topic_counts), startangle=140,
    pctdistance=0.82, labeldistance=1.12,
    wedgeprops=dict(linewidth=1.5, edgecolor='white'), textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
centre_circle = plt.Circle((0,0), 0.55, fc='white')
ax.add_artist(centre_circle)
ax.text(0, 0, f'N={len(df)}\nSermons', ha='center', va='center', fontsize=12, fontweight='bold', color='#333')
ax.set_title('Fig. 5 – Thematic Distribution of Khutbah Jumat Corpus\n(keyword-rule classification)', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig('fig05_topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 5 saved')

## Section 2: Pseudo-Labeling Pipeline
This part was already computed for real in the original notebook. Fix #3
(component-score normalization) is applied here.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

ISLAMIC_KEYWORDS = [
    # Aqidah (47 terms)
    'akhirat', 'akidah', 'allah', 'aqidah', 'bidah', 'doa', 'ghaib',
    'hisab', 'ihsan', 'iman', 'islam', 'izrail', 'jahannam', 'jannah',
    'jibril', 'kafir', 'kiamat', 'kitab', 'malaikat', 'masjid', 'mizan',
    'mulhid', 'munkar', 'murtad', 'musyrik', 'nabi', 'nakir', 'neraka',
    'qada', 'qadar', 'quran', 'rasul', 'rasulullah', 'ridwan', 'risalah',
    'rukun', 'sifat', 'sunnatullah', 'surga', 'syafaat', 'syirik', 'takdir',
    'tauhid', 'tauhidullah', 'tawakal', 'wahyu', 'yakin',
    # Fiqh (62 terms)
    'adzan', 'akad', 'dam', 'dhuha', 'fikih', 'fiqih', 'hadas',
    'haji', 'halal', 'haram', 'ibadah', 'ihram', 'ijab', 'imam',
    'infak', 'iqamat', 'itikaf', 'jamaah', 'jumat', 'junub', 'kabul',
    'kafarat', 'khatib', 'khutbah', 'mahar', 'mahram', 'makmum', 'makruh',
    'muadzin', 'muslimat', 'muslimin', 'najis', 'nikah', 'nisab', 'puasa',
    'qadha', 'qurban', 'rakaat', 'ramadan', 'rujuk', 'rukuk', 'sahur',
    'salam', 'salat', 'sedekah', 'shalat', 'sholat', 'sujud', 'syariah',
    'syariat', 'takbir', 'talak', 'tarawih', 'tawaf', 'umrah', 'wajib',
    'wali', 'waris', 'witir', 'wudu', 'wukuf', 'zakat',
    # Akhlak (50 terms)
    'adab', 'akhlak', 'amal', 'amanah', 'budi', 'dendam', 'dengki',
    'dermawan', 'disiplin', 'durhaka', 'dusta', 'empati', 'fitnah', 'ghibah',
    'hasad', 'hormat', 'ikhlas', 'istiqomah', 'jujur', 'karimah', 'kikir',
    'lembut', 'malu', 'namimah', 'niat', 'pemaaf', 'pemurah', 'qanaah',
    'qonaah', 'rahmah', 'rendah', 'riya', 'sabar', 'santun', 'silaturahim',
    'silaturahmi', 'sombong', 'syukur', 'tabah', 'takabbur', 'takwa', 'taqwa',
    'tawadhu', 'teladan', 'toleransi', 'ujub', 'uswah', 'wara', 'wibawa',
    'zuhud',
    # Tazkiyah (40 terms)
    'ampunan', 'barzakh', 'dosa', 'dzikir', 'futur', 'hidayah', 'husnul',
    'istighfar', 'khatimah', 'khauf', 'khusyuk', 'kubur', 'maghfirah', 'mahabbah',
    'muhasabah', 'mujahadah', 'munajat', 'muraqabah', 'mursyid', 'nafas', 'nafsu',
    'nikmat', 'pahala', 'rahmat', 'raja', 'riyadah', 'ruh', 'shaleh',
    'shalih', 'siksa', 'suul', 'tadabbur', 'tafakur', 'tahajud', 'taubat',
    'taufik', 'tobat', 'wasilah', 'wirid', 'zikir',
    # Muamalah (37 terms)
    'adil', 'bangsa', 'bank', 'bisnis', 'dagang', 'ekonomi', 'gadai',
    'gaji', 'gotong', 'harta', 'hibah', 'hutang', 'ijarah', 'investasi',
    'keadilan', 'kezaliman', 'kontrak', 'masyarakat', 'muamalah', 'musyawarah', 'negara',
    'pemimpin', 'perdagangan', 'perniagaan', 'persaudaraan', 'profesi', 'rakyat', 'riba',
    'sewa', 'tetangga', 'transaksi', 'ukhuwah', 'umat', 'upah', 'warisan',
    'wasiat', 'zalim',
]

def sentence_tokenize(text):
    sents = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [s.strip() for s in sents if len(s.split()) >= 5]

def position_score(idx, total, intro=0.15, outro=0.10):
    if total == 0:
        return 0.0
    rel = idx / total
    if rel <= intro:
        return 1.0 - rel/intro * 0.4
    elif rel >= (1 - outro):
        return 0.6 + (rel - (1-outro))/outro * 0.4
    else:
        return 0.2

def keyword_density(sentence):
    tokens = sentence.lower().split()
    if not tokens:
        return 0.0
    hits = sum(1 for t in tokens if any(kw in t for kw in ISLAMIC_KEYWORDS))
    return hits / len(tokens)

all_sentences = []
doc_sentences = []
for text in df['transcript_clean']:
    sents = sentence_tokenize(text)
    doc_sentences.append(sents)
    all_sentences.extend(sents)

print(f'Total sentences: {len(all_sentences):,}  (was hardcoded as \'38,247\' in original Fig. 5 title)')

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(all_sentences)
tfidf_scores_all = np.array(tfidf_matrix.mean(axis=1)).flatten()

ALPHA, BETA, GAMMA = 0.5, 0.3, 0.2
THRESHOLD_PERCENTILE = 70

records = []
sent_offset = 0
for doc_idx, (sents, topic) in enumerate(zip(doc_sentences, df['topic'])):
    n = len(sents)
    if n == 0:
        continue
    for i, sent in enumerate(sents):
        tfidf_s = float(tfidf_scores_all[sent_offset + i])
        pos_s = position_score(i, n)
        kw_s = keyword_density(sent)
        records.append({'doc_idx': doc_idx, 'sent_idx': i, 'sentence': sent,
                        'tfidf': tfidf_s, 'position': pos_s, 'keyword': kw_s, 'topic': topic})
    sent_offset += n

sent_df = pd.DataFrame(records)

# FIX #3: normalize each component to [0,1] before combining. Raw tfidf
# (mean ~0.0004) is ~1000x smaller than position (0.2-1.0) and ~100x
# smaller than keyword density (0-0.8) -- without normalization, alpha
# has almost no real influence on ranking regardless of its nominal value.
for col in ['tfidf', 'position', 'keyword']:
    cmin, cmax = sent_df[col].min(), sent_df[col].max()
    rng = cmax - cmin if cmax > cmin else 1.0
    sent_df[col + '_norm'] = (sent_df[col] - cmin) / rng

sent_df['composite'] = ALPHA*sent_df['tfidf_norm'] + BETA*sent_df['position_norm'] + GAMMA*sent_df['keyword_norm']

def label_per_doc(grp):
    thresh = grp['composite'].quantile(THRESHOLD_PERCENTILE/100)
    grp = grp.copy()
    grp['label'] = (grp['composite'] >= thresh).astype(int)
    return grp

sent_df = sent_df.groupby('doc_idx', group_keys=False)[sent_df.columns].apply(label_per_doc).reset_index(drop=True)
print(f'Total labeled sentences: {len(sent_df):,}')
print(f'Positive rate: {sent_df["label"].mean():.1%}')
sent_df.head()

## Figure 6: Pseudo-Label Score Distribution\nTitle now uses the real sentence count instead of the hardcoded '38,247'.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for lbl, color, name in [(0, PALETTE[3], 'Non-summary (0)'), (1, PALETTE[0], 'Summary-worthy (1)')]:
    axes[0].hist(sent_df.loc[sent_df['label']==lbl, 'composite'], bins=40,
                 alpha=0.7, color=color, label=name, edgecolor='white')
thresh_val = sent_df.groupby('doc_idx')['composite'].quantile(0.7).mean()
axes[0].axvline(thresh_val, color='black', ls='--', lw=2, label=f'Threshold ≈ {thresh_val:.3f}')
axes[0].set_xlabel('Composite Pseudo-Label Score (normalized)')
axes[0].set_ylabel('Number of Sentences')
axes[0].set_title('(a) Score Distribution by Label')
axes[0].legend()

for comp, color in [('tfidf_norm', PALETTE[0]), ('position_norm', PALETTE[1]), ('keyword_norm', PALETTE[2])]:
    axes[1].hist(sent_df[comp], bins=30, alpha=0.6, color=color, label=comp.replace('_norm','').upper(), edgecolor='white')
axes[1].set_xlabel('Component Score Value (normalized)')
axes[1].set_ylabel('Number of Sentences')
axes[1].set_title('(b) Individual Component Score Distributions')
axes[1].legend()

fig.suptitle(f'Fig. 6 – Pseudo-Label Score Distribution Across {len(sent_df):,} Sentence Instances', fontsize=14)
plt.tight_layout()
plt.savefig('fig06_pseudolabel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 6 saved')

## Figure 7: Relative Contribution of Pseudo-Labeling Components

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Mean weighted contribution of each component to the composite score
mean_contribs = {
    'TF-IDF\n(\u03b1\u00d7norm)': (ALPHA * sent_df['tfidf_norm']).mean(),
    'Position\n(\u03b2\u00d7norm)': (BETA * sent_df['position_norm']).mean(),
    'Keyword\n(\u03b3\u00d7norm)': (GAMMA * sent_df['keyword_norm']).mean(),
}
bars = axes[0].bar(mean_contribs.keys(), mean_contribs.values(),
                    color=[PALETTE[0], PALETTE[1], PALETTE[2]], edgecolor='white')
for b, v in zip(bars, mean_contribs.values()):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Mean Weighted Contribution to Composite Score')
axes[0].set_title('(a) Average Weighted Contribution per Component')

# (b) Correlation of each component's raw (normalized) score with the final positive label,
# as a complementary view of how much each signal aligns with what actually gets selected
corrs = {
    'TF-IDF': sent_df['tfidf_norm'].corr(sent_df['label']),
    'Position': sent_df['position_norm'].corr(sent_df['label']),
    'Keyword': sent_df['keyword_norm'].corr(sent_df['label']),
}
bars2 = axes[1].bar(corrs.keys(), corrs.values(), color=[PALETTE[0], PALETTE[1], PALETTE[2]], edgecolor='white')
for b, v in zip(bars2, corrs.values()):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel("Point-Biserial Correlation with Positive Label")
axes[1].set_title('(b) Component Alignment with Final Pseudo-Label')
axes[1].axhline(0, color='black', lw=0.8)

fig.suptitle('Fig. 7 \u2013 Relative Contribution of TF-IDF, Position, and Keyword Density in Pseudo-Labeling', fontsize=13)
plt.tight_layout()
plt.savefig('fig07_component_contribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 7 saved')
print('Mean weighted contributions:', {k: round(v,4) for k,v in mean_contribs.items()})
print('Correlations with positive label:', {k: round(v,4) for k,v in corrs.items()})


## Weight Sensitivity Analysis (fix #4)
Original judged each (alpha,beta,gamma) config by the average of its own
composite score (circular). This version scores each config by real
ROUGE-1 of its top-30% extraction against an external LexRank reference,
on a 20-document held-out validation set distinct from the 21-doc test set.

**Note:** this still uses an *unsupervised* pseudo-reference, so it does
not by itself resolve the evaluation-circularity concern raised by
Reviewer 2 -- that requires the human-annotated reference set described
in the Reviewer 2 response. It only fixes the mechanical bug where the
metric measured nothing external.

In [ ]:
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer as rouge_lib

def tfidf_extractive_summary(text, ratio=0.30):
    """Rank sentences by their own mean TF-IDF weight."""
    sents = sentence_tokenize(text)
    if len(sents) < 3:
        return ' '.join(sents)
    tv = TfidfVectorizer()
    try:
        X = tv.fit_transform(sents)
        scores = np.array(X.mean(axis=1)).flatten()
    except ValueError:
        scores = np.arange(len(sents), 0, -1).astype(float)
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(scores)[-k:])
    return ' '.join([sents[i] for i in top_idx])

def textrank_summary(text, ratio=0.30, damping=0.85):
    """Genuine TextRank: PageRank over a cosine-similarity sentence graph."""
    sents = sentence_tokenize(text)
    if len(sents) < 3:
        return ' '.join(sents)
    tv = TfidfVectorizer()
    try:
        X = tv.fit_transform(sents)
        sim = cosine_similarity(X)
        np.fill_diagonal(sim, 0)
        graph = nx.from_numpy_array(sim)
        pr = nx.pagerank(graph, alpha=damping, max_iter=200)
        scores = np.array([pr[i] for i in range(len(sents))])
    except Exception:
        scores = np.ones(len(sents))
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(scores)[-k:])
    return ' '.join([sents[i] for i in top_idx])

def lexrank_summary(text, ratio=0.30, threshold=0.1):
    """Genuine LexRank: eigenvector centrality on a thresholded similarity graph."""
    sents = sentence_tokenize(text)
    if len(sents) < 3:
        return ' '.join(sents)
    tv = TfidfVectorizer()
    try:
        X = tv.fit_transform(sents)
        sim = cosine_similarity(X)
        np.fill_diagonal(sim, 0)
        adj = (sim >= threshold).astype(float)
        graph = nx.from_numpy_array(adj)
        centrality = nx.eigenvector_centrality_numpy(graph)
        scores = np.array([centrality[i] for i in range(len(sents))])
    except Exception:
        scores = np.ones(len(sents))
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(scores)[-k:])
    return ' '.join([sents[i] for i in top_idx])

scorer_r1 = rouge_lib.RougeScorer(['rouge1'], use_stemmer=False)

test_idx = df.index[-21:].tolist()
remaining_idx = [i for i in df.index if i not in set(test_idx)]
np.random.seed(42)
val_idx = list(np.random.choice(remaining_idx, size=20, replace=False))
val_texts = df.loc[val_idx, 'transcript_clean'].tolist()
val_refs_summaries = [lexrank_summary(t, 0.30) for t in val_texts]

weight_configs = [(0.4,0.4,0.2), (0.5,0.3,0.2), (0.6,0.2,0.2), (0.4,0.3,0.3), (0.3,0.3,0.4)]

def top_k_summary_from_scores(doc_sents, score_col, ratio=0.30):
    n = len(doc_sents)
    k = max(1, int(n * ratio))
    top_idx = doc_sents[score_col].nlargest(k).index
    picked = doc_sents.loc[sorted(top_idx)]
    return ' '.join(picked['sentence'].tolist())

sensitivity_results = []
for alpha, beta, gamma in weight_configs:
    r1_scores = []
    for vi, ref_summary in zip(val_idx, val_refs_summaries):
        doc_sents = sent_df[sent_df['doc_idx']==vi].copy()
        if len(doc_sents) == 0:
            continue
        doc_sents['composite_test'] = alpha*doc_sents['tfidf_norm'] + beta*doc_sents['position_norm'] + gamma*doc_sents['keyword_norm']
        cand_summary = top_k_summary_from_scores(doc_sents, 'composite_test', 0.30)
        if not cand_summary.strip() or not ref_summary.strip():
            continue
        sc = scorer_r1.score(ref_summary, cand_summary)
        r1_scores.append(sc['rouge1'].fmeasure)
    sensitivity_results.append({'alpha': alpha, 'beta': beta, 'gamma': gamma,
                                 'val_rouge1_mean': float(np.mean(r1_scores)), 'n_docs': len(r1_scores)})

sensitivity_df = pd.DataFrame(sensitivity_results).sort_values('val_rouge1_mean', ascending=False)
print('Weight Sensitivity Analysis (real ROUGE-1 on held-out validation set):')
sensitivity_df

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
labels = [f"α={a}\nβ={b}\nγ={g}" for a,b,g in zip(sensitivity_df['alpha'], sensitivity_df['beta'], sensitivity_df['gamma'])]
ax.bar(labels, sensitivity_df['val_rouge1_mean'], edgecolor='black', color=PALETTE[0])
ax.set_ylabel('Validation ROUGE-1 (vs. LexRank reference)')
ax.set_title('Weight Sensitivity Analysis (real, externally-validated)')
plt.tight_layout()
plt.show()

## Section 3: IndoBERT Fine-Tuning (GPU REQUIRED)
This section is largely unchanged from the original -- it was already a
real, correctly-implemented fine-tuning loop (assuming it is actually run
with `ACTUAL_TRAINING=True` on a GPU, rather than falling through to the
simulated-curve branch). **The simulated-curve fallback has been removed
from this corrected notebook** so results can never silently be fake.

**Run this cell in Google Colab with a T4 GPU.** It will not run in a
CPU-only / no-internet environment because it needs to download
`indobenchmark/indobert-base-p1` from Hugging Face.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentenceDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], max_length=MAX_LEN, truncation=True,
                        padding='max_length', return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Use only TRAIN + VAL documents for fine-tuning; the 21-doc test set and
# 20-doc weight-sensitivity validation set are excluded to avoid leakage.
excluded_idx = set(test_idx) | set(val_idx)
train_pool = sent_df[~sent_df['doc_idx'].isin(excluded_idx)]

texts_all = train_pool['sentence'].tolist()
labels_all = train_pool['label'].tolist()
X_tr, X_val, y_tr, y_val = train_test_split(texts_all, labels_all, test_size=0.2,
                                              random_state=42, stratify=labels_all)

train_ds = SentenceDataset(X_tr, y_tr)
val_ds = SentenceDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'This cell requires a GPU runtime (Runtime > Change runtime type > T4 GPU).'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
learning_rates, val_f1s = [], []  # tracked for Fig. 10
from sklearn.metrics import f1_score

for epoch in range(EPOCHS):
    model.train()
    ep_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        lbls = batch['label'].to(device)
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])  # tracked for Fig. 10
        ep_loss += out.loss.item()
        preds = out.logits.argmax(dim=-1)
        correct += (preds == lbls).sum().item()
        total += len(lbls)
    train_losses.append(ep_loss/len(train_loader))
    train_accs.append(correct/total)

    model.eval()
    vl, vc, vt = 0, 0, 0
    all_val_preds, all_val_lbls = [], []  # tracked for Fig. 10 (F1)
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)
            out = model(input_ids=input_ids, attention_mask=attn_mask, labels=lbls)
            vl += out.loss.item()
            preds = out.logits.argmax(dim=-1)
            vc += (preds == lbls).sum().item()
            vt += len(lbls)
            all_val_preds.extend(preds.cpu().numpy().tolist())
            all_val_lbls.extend(lbls.cpu().numpy().tolist())
    val_losses.append(vl/len(val_loader))
    val_accs.append(vc/vt)
    val_f1s.append(f1_score(all_val_lbls, all_val_preds, average='binary'))
    print(f'Epoch {epoch+1}/{EPOCHS} | TrainLoss={train_losses[-1]:.4f} ValLoss={val_losses[-1]:.4f} ValAcc={val_accs[-1]:.4f} ValF1={val_f1s[-1]:.4f}')

model.save_pretrained('/content/indobert_khutbah_finetuned')
tokenizer.save_pretrained('/content/indobert_khutbah_finetuned')
print('✅ Real fine-tuning complete, model saved to /content/indobert_khutbah_finetuned')

## Figure 8: Training & Validation Loss

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
epochs_x = list(range(1, len(train_losses)+1))
ax.plot(epochs_x, train_losses, marker='o', label='Train Loss', color=PALETTE[0], lw=2)
ax.plot(epochs_x, val_losses, marker='s', label='Validation Loss', color=PALETTE[3], lw=2)
best_epoch = int(np.argmin(val_losses)) + 1
ax.axvline(best_epoch, color='gray', ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Fig. 8 – Training and Validation Loss During IndoBERT Fine-Tuning')
ax.legend()
plt.tight_layout()
plt.savefig('fig08_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 8 saved')

## Figure 9: Training & Validation Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(epochs_x, train_accs, marker='o', label='Train Accuracy', color=PALETTE[0], lw=2)
ax.plot(epochs_x, val_accs, marker='s', label='Validation Accuracy', color=PALETTE[3], lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Fig. 9 – Training and Validation Accuracy During IndoBERT Fine-Tuning')
ax.legend()
plt.tight_layout()
plt.savefig('fig09_training_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 9 saved')

## Figure 10: Learning Rate Schedule and Validation F1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(learning_rates)+1), learning_rates, color=PALETTE[0], lw=1.5)
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Learning Rate')
axes[0].set_title('(a) Learning Rate Schedule (Warm-up + Linear Decay)')
warmup_steps_actual = int(0.1 * len(learning_rates))
axes[0].axvline(warmup_steps_actual, color='gray', ls='--', alpha=0.7, label=f'End of warm-up (step {warmup_steps_actual})')
axes[0].legend()

axes[1].plot(epochs_x, val_f1s, marker='o', color=PALETTE[3], lw=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation F1 (positive class)')
axes[1].set_title('(b) Validation F1 Progression')
best_f1_epoch = int(np.argmax(val_f1s)) + 1
axes[1].axvline(best_f1_epoch, color='gray', ls='--', alpha=0.7, label=f'Best F1 (epoch {best_f1_epoch})')
axes[1].legend()

fig.suptitle('Fig. 10 \u2013 Learning Rate Schedule and Validation Performance During IndoBERT Fine-Tuning', fontsize=13)
plt.tight_layout()
plt.savefig('fig10_lr_schedule_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\u2705 Figure 10 saved. Peak validation F1 = {max(val_f1s):.4f} at epoch {best_f1_epoch}')


## Section 4B: Real Evaluation (GPU / internet REQUIRED)
Replaces every hardcoded number in the original Section 4 with real
computation:
- **Real IndoBERT inference** from the checkpoint trained above (fix #8)
- **Real BERTSum baseline** using a genuine BERT sentence encoder, not TF-IDF (fix #7)
- **Real BERTScore** via the `bert-score` package (fix #10)
- **Real confusion matrix** from actual model predictions on the test set (fix #9)
- **Real topic-wise ROUGE** computed per topic subset (fix #9)
- **Real ablation study** -- retrain/re-score with components removed (fix #9)
- **Real compression-ratio curve** -- re-evaluate at each ratio (fix #9)
- **Real qualitative examples** pulled from actual test-set outputs (fix #11 continuation)

In [ ]:
import torch
from bert_score import score as bertscore_fn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Load the fine-tuned model from Section 3 ---
tokenizer = AutoTokenizer.from_pretrained('/content/indobert_khutbah_finetuned')
finetuned_model = AutoModelForSequenceClassification.from_pretrained('/content/indobert_khutbah_finetuned').to(device)
finetuned_model.eval()

def indobert_summary_real(text, ratio=0.30, max_len=128):
    """FIX #8: real inference from the fine-tuned model (was: proxy composite score)."""
    sents = sentence_tokenize(text)
    if len(sents) < 3:
        return ' '.join(sents), np.array([])
    probs = []
    with torch.no_grad():
        for i in range(0, len(sents), 32):
            batch = sents[i:i+32]
            enc = tokenizer(batch, max_length=max_len, truncation=True, padding=True, return_tensors='pt').to(device)
            logits = finetuned_model(**enc).logits
            p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()  # P(summary-worthy)
            probs.extend(p.tolist())
    probs = np.array(probs)
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(probs)[-k:])
    return ' '.join([sents[i] for i in top_idx]), probs

# --- Real BERTSum-style baseline: generic (non-fine-tuned) BERT encoder + centrality ---
generic_tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
generic_model = AutoModel.from_pretrained('bert-base-multilingual-cased').to(device)
generic_model.eval()

def bertsum_generic_summary(text, ratio=0.30):
    """FIX #7: real generic-BERT extractive baseline (was: TF-IDF renamed 'BERTSum')."""
    sents = sentence_tokenize(text)
    if len(sents) < 3:
        return ' '.join(sents)
    embs = []
    with torch.no_grad():
        for i in range(0, len(sents), 32):
            batch = sents[i:i+32]
            enc = generic_tokenizer(batch, max_length=128, truncation=True, padding=True, return_tensors='pt').to(device)
            out = generic_model(**enc).last_hidden_state[:, 0, :]  # [CLS] embedding
            embs.append(out.cpu().numpy())
    embs = np.concatenate(embs, axis=0)
    from sklearn.metrics.pairwise import cosine_similarity
    sim = cosine_similarity(embs)
    np.fill_diagonal(sim, 0)
    centrality = sim.sum(axis=1)  # degree centrality in embedding space
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(centrality)[-k:])
    return ' '.join([sents[i] for i in top_idx])

print('✅ Real models loaded: fine-tuned IndoBERT + generic multilingual BERT for BERTSum baseline')

In [ ]:
# --- HUMAN REFERENCE HOOK ---
# Per the Reviewer 2 response, the authoritative evaluation should use a
# human-annotated reference for the 21 test documents, not an unsupervised
# pseudo-reference. If you have collected human annotations, put them in a
# CSV with columns [doc_idx, human_reference_summary] and load them here:
HUMAN_REFERENCE_CSV = None  # e.g. '/content/human_annotated_references.csv'

if HUMAN_REFERENCE_CSV is not None:
    human_refs_df = pd.read_csv(HUMAN_REFERENCE_CSV).set_index('doc_idx')
    refs_summaries = [human_refs_df.loc[i, 'human_reference_summary'] for i in test_idx]
    print(f'✅ Using {len(refs_summaries)} HUMAN-annotated references (resolves Reviewer 2 Comment 1)')
else:
    refs_summaries = [lexrank_summary(t, 0.30) for t in df.loc[test_idx, 'transcript_clean']]
    print('⚠️  Using LexRank PSEUDO-reference -- evaluation-circularity concern (Reviewer 2, Comment 1) '
          'is NOT resolved until HUMAN_REFERENCE_CSV is provided.')

test_texts = df.loc[test_idx, 'transcript_clean'].tolist()
test_topics = df.loc[test_idx, 'topic'].tolist()

print('Generating summaries with real methods on the test set...')
tfidf_summaries = [tfidf_extractive_summary(t, 0.30) for t in test_texts]
textrank_summaries = [textrank_summary(t, 0.30) for t in test_texts]
lexrank_summaries = [lexrank_summary(t, 0.30) for t in test_texts]
bertsum_summaries = [bertsum_generic_summary(t, 0.30) for t in test_texts]
indobert_results = [indobert_summary_real(t, 0.30) for t in test_texts]
indobert_summaries = [r[0] for r in indobert_results]
indobert_probs = [r[1] for r in indobert_results]

def avg_rouge(systems, refs):
    r1, r2, rL = [], [], []
    for s, r in zip(systems, refs):
        if not s.strip() or not r.strip():
            continue
        sc = scorer.score(r, s)
        r1.append(sc['rouge1'].fmeasure); r2.append(sc['rouge2'].fmeasure); rL.append(sc['rougeL'].fmeasure)
    return np.mean(r1), np.mean(r2), np.mean(rL)

scorer = rouge_lib.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)

print('Computing real ROUGE + real BERTScore (fix #6, #10)...')
# NOTE: when HUMAN_REFERENCE_CSV is None, refs_summaries is built with lexrank_summary(),
# the SAME function used for the 'LexRank' baseline row below. This makes the 'LexRank'
# row circular (it will score close to 1.0, comparing itself to itself) -- the same bug
# already fixed for Table 3 / Fig. 13. We keep LexRank in `results` for completeness but
# flag it, and exclude it from the radar chart (Fig. 11) to avoid a misleading scale.
lexrank_is_circular = (HUMAN_REFERENCE_CSV is None)
if lexrank_is_circular:
    print("⚠️  'LexRank' row below is circular (reference == LexRank output) until "
          "HUMAN_REFERENCE_CSV is provided -- excluded from Fig. 11 radar chart for this reason.")

results = {}
for name, summaries in [('TF-IDF', tfidf_summaries), ('TextRank', textrank_summaries),
                         ('LexRank', lexrank_summaries), ('BERTSum (generic)', bertsum_summaries),
                         ('IndoBERT (Ours)', indobert_summaries)]:
    r1, r2, rL = avg_rouge(summaries, refs_summaries)
    P, R, F1 = bertscore_fn(summaries, refs_summaries, lang='id', verbose=False)
    results[name] = (r1, r2, rL, float(F1.mean()))

print('\n=== REAL Evaluation Results (Table 3) ===')
for method, (r1, r2, rL, bs) in results.items():
    print(f'{method:<22}: R1={r1:.4f} R2={r2:.4f} RL={rL:.4f} BERTScore={bs:.4f}')

## Figure 11: Radar Chart (Multi-Metric Comparison)

In [ ]:
from math import pi

# Exclude LexRank from the radar when it is circular (see warning above);
# always exclude it from THIS specific visualization regardless, since even a
# non-circular LexRank score is not central to the story this chart tells and
# including a near-1.0 self-referential bar would still distort readers' first
# impression if HUMAN_REFERENCE_CSV is swapped in later without re-checking this cell.
radar_methods = [m for m in results.keys() if not (lexrank_is_circular and m == 'LexRank')]
metrics_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1']
N = len(metrics_labels)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = [PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3], PALETTE[4]]
for i, method in enumerate(radar_methods):
    vals = list(results[method])
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=method, color=colors_radar[i % len(colors_radar)])
    ax.fill(angles, vals, alpha=0.08, color=colors_radar[i % len(colors_radar)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_labels)
ax.set_ylim(0, 1.0)
title_note = '' if not lexrank_is_circular else ' (LexRank excluded: circular vs. pseudo-reference)'
ax.set_title(f'Fig. 11 \u2013 Multi-Metric Comparison Across Summarization Methods{title_note}', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('fig11_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\u2705 Figure 11 saved. Methods shown: {radar_methods}')


## Section 4C: Bootstrap 95% Confidence Intervals (editor poin 1.2)
Resampling dilakukan di tingkat **dokumen** (21 dokumen test), 1.000 kali, dengan seed tetap.
CI = persentil 2,5 dan 97,5 dari rata-rata hasil resample. Untuk perbandingan antar metode
dipakai **paired bootstrap** (resample dokumen yang sama untuk kedua metode), karena ini lebih
tepat daripada sekadar melihat apakah dua CI tumpang tindih.

In [ ]:
import numpy as np, pandas as pd

N_BOOT = 1000
BOOT_SEED = 42
METHODS = [('TF-IDF', tfidf_summaries), ('TextRank', textrank_summaries),
           ('BERTSum (generic)', bertsum_summaries), ('IndoBERT (Ours)', indobert_summaries)]
METRICS = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1']

# ---- 1. Skor per dokumen (logika sama dengan avg_rouge / results di atas) ----
rows = []
for name, summaries in METHODS:
    P, R, F = bertscore_fn(summaries, refs_summaries, lang='id', verbose=False)
    F = F.cpu().numpy()
    for j, (s, r) in enumerate(zip(summaries, refs_summaries)):
        if s.strip() and r.strip():
            sc = scorer.score(r, s)
            r1, r2, rl = sc['rouge1'].fmeasure, sc['rouge2'].fmeasure, sc['rougeL'].fmeasure
        else:  # avg_rouge() melewati pasangan kosong -> NaN di sini, diabaikan saat rata-rata
            r1 = r2 = rl = np.nan
        rows.append({'method': name, 'doc_idx': test_idx[j], 'topic': test_topics[j],
                     'ROUGE-1': r1, 'ROUGE-2': r2, 'ROUGE-L': rl, 'BERTScore F1': float(F[j])})
per_doc = pd.DataFrame(rows)
per_doc.to_csv('per_document_scores.csv', index=False)

# ---- 2. Bootstrap CI per metode x metrik ----
rng = np.random.default_rng(BOOT_SEED)
n_docs = len(test_idx)
boot_idx = rng.integers(0, n_docs, size=(N_BOOT, n_docs))   # indeks resample dipakai bersama semua metode

def wide(metric):
    return per_doc.pivot(index='doc_idx', columns='method', values=metric).loc[test_idx]

table = []
for name, _ in METHODS:
    row = {'Method': name}
    for m in METRICS:
        v = wide(m)[name].to_numpy()
        boots = np.array([np.nanmean(v[b]) for b in boot_idx])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        row[m] = np.nanmean(v); row[m + ' CI low'] = lo; row[m + ' CI high'] = hi
    table.append(row)
ci_df = pd.DataFrame(table)
ci_df.to_csv('table3_with_ci.csv', index=False)

print(f'=== Tabel 3 dengan 95% bootstrap CI (n = {n_docs} dokumen, {N_BOOT} resample) ===')
for _, r in ci_df.iterrows():
    cells = ['%s %.4f [%.4f, %.4f]' % (m, r[m], r[m+' CI low'], r[m+' CI high']) for m in METRICS]
    print('%-18s | ' % r['Method'] + ' | '.join(cells))

# ---- 3. Cek: rata-rata di sini harus sama dengan `results` (sel Table 3 di atas) ----
for name, _ in METHODS:
    for k, m in enumerate(METRICS):
        a, b = ci_df.set_index('Method').loc[name, m], results[name][k]
        assert abs(a - b) < 1e-4, f'Tidak cocok: {name} {m}: {a} vs {b}'
print('\n✅ Rata-rata per dokumen cocok dengan tabel `results` di atas.')

# ---- 4. Bandingkan dengan Tabel 3 di naskah lama ----
PAPER_TABLE3 = {'TF-IDF': (0.4400, 0.3554, 0.3704, 0.6780),
                'TextRank': (0.4322, 0.3262, 0.3377, 0.7240),
                'BERTSum (generic)': (0.4365, 0.3047, 0.3152, 0.7407),
                'IndoBERT (Ours)': (0.4700, 0.3751, 0.3982, 0.7194)}
diff_found = False
for name, vals in PAPER_TABLE3.items():
    for k, m in enumerate(METRICS):
        new = ci_df.set_index('Method').loc[name, m]
        if abs(new - vals[k]) > 0.0005:
            diff_found = True
            print(f'  BEDA: {name:<18} {m:<13} naskah={vals[k]:.4f}  run ini={new:.4f}')
print('⚠️  Ada angka yang berbeda dari naskah -> Tabel 3 diganti SELURUHNYA dengan angka run ini.'
      if diff_found else '✅ Semua rata-rata sama dengan Tabel 3 di naskah.')

In [ ]:
# ---- Paired bootstrap: selisih IndoBERT - baseline (dokumen yang sama) ----
pairs = []
for base, _ in METHODS[:-1]:
    for m in METRICS:
        w = wide(m)
        d = (w['IndoBERT (Ours)'] - w[base]).to_numpy()
        boots = np.array([np.nanmean(d[b]) for b in boot_idx])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        pairs.append({'Comparison': f'IndoBERT - {base}', 'Metric': m, 'Mean diff': np.nanmean(d),
                      'CI low': lo, 'CI high': hi, 'CI excludes 0': not (lo <= 0 <= hi)})
paired_df = pd.DataFrame(pairs)
paired_df.to_csv('paired_differences.csv', index=False)
print('=== Paired bootstrap 95% CI untuk selisih (IndoBERT - baseline) ===')
print(paired_df.to_string(index=False, float_format='{:.4f}'.format))
print('\n"CI excludes 0 = True" -> selisih konsisten; False -> tidak bisa diklaim lebih unggul.')

## Figure 11 (baru): Grouped Bar Chart dengan 95% CI (editor poin 2.1)
Menggantikan radar chart. ROUGE dan BERTScore dipisah ke dua panel karena skalanya berbeda.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [3, 1.3]})
names = ci_df['Method'].tolist()
colors = [PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3]]
w = 0.8 / len(names)

for ax, mets in [(axes[0], ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']), (axes[1], ['BERTScore F1'])]:
    x = np.arange(len(mets))
    for i, name in enumerate(names):
        r = ci_df.set_index('Method').loc[name]
        means = np.array([r[m] for m in mets])
        err = np.array([[r[m] - r[m+' CI low'] for m in mets], [r[m+' CI high'] - r[m] for m in mets]])
        ax.bar(x + (i - (len(names)-1)/2) * w, means, w, yerr=err, capsize=3,
               label=name, color=colors[i], edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(mets)
    ax.set_ylabel('F1 score'); ax.set_ylim(0, 1)
axes[0].set_title('(a) ROUGE (vs. LexRank pseudo-reference)')
axes[1].set_title('(b) BERTScore F1')
axes[0].legend(loc='upper right', fontsize=9)
fig.suptitle('Fig. 11 – Comparison of summarization methods on the 21-document test set '
             '(error bars: 95% bootstrap CI)', fontsize=12)
plt.tight_layout()
plt.savefig('fig11_grouped_bar_ci.png', dpi=300, bbox_inches='tight')
plt.show()

## Uji statistik ROUGE-1 per topik (editor poin 2.5)
Kruskal–Wallis (non-parametrik) pada ROUGE-1 per dokumen IndoBERT, dikelompokkan per topik.

In [ ]:
from scipy.stats import kruskal
ib = per_doc[per_doc['method'] == 'IndoBERT (Ours)'].dropna(subset=['ROUGE-1'])
groups = {t: g['ROUGE-1'].to_numpy() for t, g in ib.groupby('topic')}
print({t: len(v) for t, v in groups.items()})
valid = [v for v in groups.values() if len(v) >= 2]
H, p = kruskal(*valid)
print(f'Kruskal-Wallis: H = {H:.3f}, p = {p:.4f}  (topik dengan n>=2: {len(valid)})')
print('p >= 0.05 -> perbedaan antar topik tidak signifikan secara statistik (analisis bersifat eksploratif).')
with open('topic_kruskal.txt', 'w') as f:
    f.write(f'n per topic: { {t: len(v) for t, v in groups.items()} }\nH={H:.4f} p={p:.4f}\n')

In [ ]:
try:
    from google.colab import files
    for f in ['table3_with_ci.csv', 'per_document_scores.csv', 'paired_differences.csv',
              'fig11_grouped_bar_ci.png', 'topic_kruskal.txt']:
        files.download(f)
except Exception as e:
    print('Download manual dari panel Files:', e)

## Figure 12: Confusion Matrix (real predictions)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

test_sent_df = sent_df[sent_df['doc_idx'].isin(test_idx)].copy()
y_true = test_sent_df['label'].values

y_pred = []
with torch.no_grad():
    for i in range(0, len(test_sent_df), 32):
        batch = test_sent_df['sentence'].iloc[i:i+32].tolist()
        enc = tokenizer(batch, max_length=128, truncation=True, padding=True, return_tensors='pt').to(device)
        logits = finetuned_model(**enc).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        y_pred.extend(preds.tolist())
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-summary','Summary'], yticklabels=['Non-summary','Summary'],
            annot_kws={'size':16,'weight':'bold'}, linewidths=2, linecolor='white')
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label'); axes[0].set_title('(a) Confusion Matrix (Counts)')
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Non-summary','Summary'], yticklabels=['Non-summary','Summary'],
            annot_kws={'size':14}, linewidths=2, linecolor='white')
axes[1].set_xlabel('Predicted Label'); axes[1].set_ylabel('True Label'); axes[1].set_title('(b) Confusion Matrix (Normalized)')
fig.suptitle('Fig. 12 – Confusion Matrix for Binary Sentence Classification (Test Set)', fontsize=14)
plt.tight_layout()
plt.savefig('fig12_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_true, y_pred, target_names=['Non-summary','Summary']))
print('✅ Figure 12 saved (from REAL model predictions, not injected noise)')

## Figure 13: ROUGE-1 by Topic Category (real per-topic evaluation)

In [ ]:
topic_rouge = {}
for topic in df['topic'].unique():
    topic_test_idx = [i for i in test_idx if df.loc[i, 'topic'] == topic]
    if len(topic_test_idx) == 0:
        continue
    idxs_in_test = [test_idx.index(i) for i in topic_test_idx]
    topic_rouge[topic] = {}
    for name, summaries in [('TF-IDF', tfidf_summaries), ('TextRank', textrank_summaries),
                             ('LexRank', lexrank_summaries), ('BERTSum', bertsum_summaries),
                             ('IndoBERT', indobert_summaries)]:
        subset_sys = [summaries[i] for i in idxs_in_test]
        subset_ref = [refs_summaries[i] for i in idxs_in_test]
        r1, _, _ = avg_rouge(subset_sys, subset_ref)
        topic_rouge[topic][name] = r1

topic_df_rouge = pd.DataFrame(topic_rouge).T
print(f'NOTE: with only 21 test documents split across 5 topics, per-topic n is small '
      f'({df.loc[test_idx,"topic"].value_counts().to_dict()}) -- interpret with caution.')
print(topic_df_rouge.round(3))

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(topic_df_rouge)); w = 0.14
for i, col in enumerate(topic_df_rouge.columns):
    ax.bar(x + (i-2)*w, topic_df_rouge[col], w, label=col, color=PALETTE[i], alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(topic_df_rouge.index, fontsize=12)
ax.set_ylabel('ROUGE-1 F1 Score')
ax.set_title('Fig. 13 – ROUGE-1 Scores by Sermon Topic Category Across All Methods (real, n varies by topic)')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('fig13_rouge_by_topic.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 13 saved (real per-topic evaluation, not hand-typed)')

## Figure 14: Ablation Study (real, CPU-based on the composite scoring formula)

In [ ]:
"""
Ablation study: measures the contribution of each pseudo-labeling component
by removing it from the composite score entirely (weight -> 0, remaining
weights renormalized to sum to 1) and re-scoring extraction via ROUGE-1
against the LexRank reference on the same 20-document validation set used
in the weight sensitivity analysis. This evaluates the pseudo-labeling
SCORING FORMULA directly (the same ranking logic that produces the
extracted summary), rather than requiring separate IndoBERT retraining
for each ablation configuration, which would need ~4x the GPU time.
"""
ablation_configs = {
    'Full Model (\u03b1=0.5,\u03b2=0.3,\u03b3=0.2)': (0.5, 0.3, 0.2),
    'No TF-IDF (\u03b2=0.6,\u03b3=0.4)': (0.0, 0.6, 0.4),
    'No Position (\u03b1=0.625,\u03b3=0.375)': (0.625, 0.0, 0.375),
    'No Keyword (\u03b1=0.625,\u03b2=0.375)': (0.625, 0.375, 0.0),
}

ablation_results = []
for label, (a, b, g) in ablation_configs.items():
    r1_scores = []
    for vi, ref_summary in zip(val_idx, val_refs_summaries):
        doc_sents = sent_df[sent_df['doc_idx'] == vi].copy()
        if len(doc_sents) == 0:
            continue
        doc_sents['ablation_score'] = a * doc_sents['tfidf_norm'] + b * doc_sents['position_norm'] + g * doc_sents['keyword_norm']
        cand_summary = top_k_summary_from_scores(doc_sents, 'ablation_score', 0.30)
        if not cand_summary.strip() or not ref_summary.strip():
            continue
        sc = scorer_r1.score(ref_summary, cand_summary)
        r1_scores.append(sc['rouge1'].fmeasure)
    ablation_results.append({'config': label, 'rouge1': float(np.mean(r1_scores))})

ablation_df = pd.DataFrame(ablation_results)
full_score = ablation_df.loc[ablation_df['config'].str.startswith('Full'), 'rouge1'].values[0]
ablation_df['delta_vs_full'] = ablation_df['rouge1'] - full_score
print('Ablation results (real ROUGE-1 on 20-doc validation set, vs LexRank reference):')
print(ablation_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5.5))
colors_ab = [PALETTE[0] if 'Full' in c else PALETTE[3] for c in ablation_df['config']]
bars = ax.bar(ablation_df['config'], ablation_df['rouge1'], color=colors_ab, edgecolor='white')
for b, v, d in zip(bars, ablation_df['rouge1'], ablation_df['delta_vs_full']):
    label_txt = f'{v:.3f}' if d == 0 else f'{v:.3f}\n({d:+.3f})'
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, label_txt, ha='center', fontsize=10)
ax.set_ylabel('ROUGE-1 (validation set, vs LexRank reference)')
ax.set_title('Fig. 14 \u2013 Ablation Study: Impact of Removing Each Pseudo-Labeling Component')
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.savefig('fig14_ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 14 saved')


## Figure 15: Compression Ratio Analysis

In [ ]:
"""
Compression ratio analysis: re-evaluates ROUGE-1/2 for the fine-tuned
IndoBERT model and the TF-IDF baseline across a range of compression
ratios (10-50%), using probabilities/scores already computed on the test
set (indobert_probs from Section 4B). This is a real re-evaluation, not
a hardcoded curve.
"""
def extract_at_ratio(sents, scores, ratio):
    if len(scores) == 0 or len(sents) == 0:
        return ''
    k = max(1, int(len(sents) * ratio))
    top_idx = sorted(np.argsort(scores)[-k:])
    return ' '.join([sents[i] for i in top_idx if i < len(sents)])

ratios = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
compression_results = {'IndoBERT (Ours)': [], 'TF-IDF': []}

test_sents_list = [sentence_tokenize(t) for t in test_texts]
tfidf_scores_list = []
for t, sents in zip(test_texts, test_sents_list):
    if len(sents) < 3:
        tfidf_scores_list.append(np.array([]))
        continue
    tv_local = TfidfVectorizer()
    try:
        X_local = tv_local.fit_transform(sents)
        tfidf_scores_list.append(np.array(X_local.mean(axis=1)).flatten())
    except ValueError:
        tfidf_scores_list.append(np.arange(len(sents), 0, -1).astype(float))

for ratio in ratios:
    indobert_cands = [extract_at_ratio(s, p, ratio) for s, p in zip(test_sents_list, indobert_probs)]
    tfidf_cands = [extract_at_ratio(s, sc, ratio) for s, sc in zip(test_sents_list, tfidf_scores_list)]
    r1_ib, r2_ib, _ = avg_rouge(indobert_cands, refs_summaries)
    r1_tf, r2_tf, _ = avg_rouge(tfidf_cands, refs_summaries)
    compression_results['IndoBERT (Ours)'].append((r1_ib, r2_ib))
    compression_results['TF-IDF'].append((r1_tf, r2_tf))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, metric_name in enumerate(['ROUGE-1', 'ROUGE-2']):
    for method, color in [('IndoBERT (Ours)', PALETTE[4]), ('TF-IDF', PALETTE[0])]:
        vals = [v[i] for v in compression_results[method]]
        axes[i].plot([r*100 for r in ratios], vals, marker='o', label=method, color=color, lw=2)
    axes[i].set_xlabel('Compression Ratio (%)')
    axes[i].set_ylabel(f'{metric_name} F1')
    axes[i].set_title(f'({chr(97+i)}) {metric_name} vs. Compression Ratio')
    axes[i].legend()
    axes[i].axvline(30, color='gray', ls=':', alpha=0.6)

fig.suptitle('Fig. 15 \u2013 Effect of Summary Compression Ratio on ROUGE-1 and ROUGE-2', fontsize=13)
plt.tight_layout()
plt.savefig('fig15_compression_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

peak_ratio_idx = int(np.argmax([v[0] for v in compression_results['IndoBERT (Ours)']]))
print(f'\u2705 Figure 15 saved. IndoBERT peak ROUGE-1 at compression ratio = {ratios[peak_ratio_idx]*100:.0f}%')


## Figure 16: Qualitative Comparison (real test-set outputs)

In [ ]:
"""
Qualitative comparison: shows REAL system outputs for a few test-set
documents (not hand-written illustrative examples), comparing the
reference, IndoBERT, and LexRank summaries side by side.
"""
n_examples = min(3, len(test_idx))
example_positions = np.linspace(0, len(test_idx) - 1, n_examples).astype(int)

fig_text_lines = []
for rank, pos in enumerate(example_positions):
    doc_id = test_idx[pos]
    topic = test_topics[pos]
    ref = refs_summaries[pos]
    ib = indobert_summaries[pos]
    lr = lexrank_summaries[pos]
    label = chr(65 + rank)  # A, B, C

    def trunc(s, n=220):
        s = s.strip().replace('\n', ' ')
        return (s[:n] + '...') if len(s) > n else s

    block = (
        f"Example {label} (topic: {topic}, doc_idx={doc_id})\n"
        f"  Reference : {trunc(ref)}\n"
        f"  IndoBERT  : {trunc(ib)}\n"
        f"  LexRank   : {trunc(lr)}\n"
    )
    fig_text_lines.append(block)
    print(block)

fig, ax = plt.subplots(figsize=(13, 2 + 2.6 * n_examples))
ax.axis('off')
full_text = '\n'.join(fig_text_lines)
ax.text(0.01, 0.99, full_text, transform=ax.transAxes, va='top', ha='left',
        fontsize=10, family='monospace', wrap=True)
ax.set_title('Fig. 16 \u2013 Qualitative Comparison of System Summaries for Representative Sermon Topics (real test-set outputs)',
              fontsize=12, loc='left')
plt.tight_layout()
plt.savefig('fig16_qualitative_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 16 saved (real outputs from the test set, not hand-written examples)')


## Section 5: Summary & Export
Prints the final real results table and packages all figures.

In [ ]:
res_df = pd.DataFrame({
    'Method': list(results.keys()),
    'ROUGE-1': [v[0] for v in results.values()],
    'ROUGE-2': [v[1] for v in results.values()],
    'ROUGE-L': [v[2] for v in results.values()],
    'BERTScore': [v[3] for v in results.values()],
}).sort_values('ROUGE-1', ascending=False).reset_index(drop=True)
print('=== FINAL REAL RESULTS (Table 3) ===')
print(res_df.to_string(index=False, float_format='{:.4f}'.format))

import zipfile, os
with zipfile.ZipFile('all_figures.zip', 'w') as zf:
    for fname in sorted(os.listdir('.')):
        if fname.startswith('fig') and fname.endswith('.png'):
            zf.write(fname)
print('\n✅ Figures zipped to all_figures.zip')
try:
    from google.colab import files
    files.download('all_figures.zip')
except Exception:
    print('Not in Colab -- find all_figures.zip in the current directory')